# Who will win? Baseline Modeling

## Loading in the data from local PostgreSQL database

In [1]:
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

In [2]:
query = """
SELECT *
FROM vw_game_model_features_v2
ORDER BY season, week;
"""

df = pd.read_sql(
    query,
    engine
)

df.shape

(26455, 47)

## Let's take a look at the data

In [4]:
print(df.shape,df.columns,sep="\n")

(26455, 47)
Index(['game_id', 'season', 'week', 'start_date', 'home_team_id', 'home_team',
       'away_team_id', 'away_team', 'home_conference', 'away_conference',
       'conference_game', 'neutral_site', 'home_pregame_elo',
       'away_pregame_elo', 'home_off_yards_per_play', 'home_off_ppa',
       'home_off_success_rate', 'home_off_explosive_rate',
       'home_off_early_down_ypp', 'home_off_third_down_rate',
       'home_def_yards_per_play_allowed', 'home_def_ppa_allowed',
       'home_def_success_rate_allowed', 'home_def_explosive_rate_allowed',
       'home_def_early_down_ypp_allowed', 'home_def_third_down_rate_allowed',
       'away_off_yards_per_play', 'away_off_ppa', 'away_off_success_rate',
       'away_off_explosive_rate', 'away_off_early_down_ypp',
       'away_off_third_down_rate', 'away_def_yards_per_play_allowed',
       'away_def_ppa_allowed', 'away_def_success_rate_allowed',
       'away_def_explosive_rate_allowed', 'away_def_early_down_ypp_allowed',
       'away_def

In [ ]:
df.info() #At the point of creating this notebook, I've only loaded play data for 2021-2025. Hence the massive amount of missing values

<class 'pandas.DataFrame'>
RangeIndex: 26455 entries, 0 to 26454
Data columns (total 47 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   game_id                           26455 non-null  int64  
 1   season                            26455 non-null  int64  
 2   week                              26455 non-null  int64  
 3   start_date                        26455 non-null  object 
 4   home_team_id                      26455 non-null  int64  
 5   home_team                         26455 non-null  str    
 6   away_team_id                      26455 non-null  int64  
 7   away_team                         26455 non-null  str    
 8   home_conference                   26376 non-null  str    
 9   away_conference                   26124 non-null  str    
 10  conference_game                   26455 non-null  bool   
 11  neutral_site                      26455 non-null  bool   
 12  home_pregame_el

In [7]:
df.head()

,game_id,season,week,start_date,home_team_id,home_team,away_team_id,away_team,home_conference,away_conference,...,away_def_third_down_rate_allowed,offensive_ppa_diff,defensive_ppa_allowed_diff,offensive_success_rate_diff,offensive_ypp_diff,pregame_elo_diff,home_points,away_points,home_point_margin,home_win
0,400795505,2015,1,2015-09-05 19:00:00-04:00,2617,Stephen F. Austin,2464,Northern Arizona,Southland,Big Sky,...,NaN,NaN,NaN,NaN,NaN,NaN,28.0,34.0,-6.0,0
1,400763497,2015,1,2015-09-05 15:30:00-04:00,218,Temple,213,Penn State,American Athletic,Big Ten,...,NaN,NaN,NaN,NaN,NaN,-67.0,27.0,10.0,17.0,1
2,400763406,2015,1,2015-09-05 15:30:00-04:00,2641,Texas Tech,2534,Sam Houston,Big 12,Southland,...,NaN,NaN,NaN,NaN,NaN,NaN,59.0,45.0,14.0,1
3,400756901,2015,1,2015-09-05 18:00:00-04:00,152,NC State,2653,Troy,ACC,Sun Belt,...,NaN,NaN,NaN,NaN,NaN,333.0,49.0,21.0,28.0,1
4,400764855,2015,1,2015-09-04 19:00:00-04:00,2567,SMU,239,Baylor,American Athletic,Big 12,...,NaN,NaN,NaN,NaN,NaN,-727.0,21.0,56.0,-35.0,0


In [8]:
df.season.value_counts().sort_index()

season
2015    1538
2016    1549
2017    1551
2018    1556
2019    1623
2020    1125
2021    2454
2022    3705
2023    3724
2024    3799
2025    3831
Name: count, dtype: int64

In [11]:
df.isna().mean().sort_values(ascending=False)

offensive_ypp_diff                  0.772822
offensive_success_rate_diff         0.772822
defensive_ppa_allowed_diff          0.772822
offensive_ppa_diff                  0.772822
away_def_early_down_ypp_allowed     0.771234
away_def_explosive_rate_allowed     0.771234
away_off_yards_per_play             0.771234
away_off_early_down_ypp             0.771234
away_off_explosive_rate             0.771234
away_off_success_rate               0.771234
away_off_ppa                        0.771234
away_def_success_rate_allowed       0.771234
away_off_third_down_rate            0.771234
away_def_ppa_allowed                0.771234
away_def_yards_per_play_allowed     0.771234
away_def_third_down_rate_allowed    0.771234
home_def_ppa_allowed                0.763599
home_def_third_down_rate_allowed    0.763599
home_def_yards_per_play_allowed     0.763561
home_def_success_rate_allowed       0.763561
home_def_early_down_ypp_allowed     0.763561
home_def_explosive_rate_allowed     0.763561
home_off_t

In [13]:
df.groupby("week")["home_off_ppa"].apply(lambda x: x.notna().mean())

week
1     0.020181
2     0.265774
3     0.255382
4     0.250000
5     0.244233
6     0.226753
7     0.230503
8     0.230216
9     0.229594
10    0.237983
11    0.252874
12    0.309524
13    0.433431
14    0.385686
15    0.263804
16    0.243902
20    0.000000
Name: home_off_ppa, dtype: float64

### The missing data from earlier seasons will be dealt with before modeling. Another thing to note is that week one will always have missing values because there is nothing to go off of yet.